# **Feature Engineering — USD/VND Volatility**

**Global Financial Shocks and USD/VND Exchange Rate Volatility**

Objective: To create a simple feature set for the research paper.
The final feature set focuses on one main problem: explaining the volatility of USD/VND using global shocks and domestic control variables.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/dataset.csv")

df.head()


,Date,usd_vnd,vix,wti_oil,us_10y,dxy,vnindex
0,2010-01-04,18474.043968,20.04,81.52,3.85,77.529999,517.05
1,2010-01-05,18469.000000,19.35,81.74,3.77,77.620003,532.53
2,2010-01-06,18469.000000,19.16,83.12,3.85,77.489998,534.46
3,2010-01-07,18474.000000,19.06,82.60,3.85,77.910004,533.34
4,2010-01-08,18469.000000,18.13,82.74,3.83,77.470001,520.90


In [2]:
clean = pd.DataFrame() 

clean["Date"] = pd.to_datetime(df["Date"])
clean["usd_vnd"] = pd.to_numeric(df["usd_vnd"], errors="coerce")
clean["vix"] = pd.to_numeric(df["vix"], errors="coerce").ffill(limit=5)
clean["us_10y"] = pd.to_numeric(df["us_10y"], errors="coerce").ffill(limit=5)
clean["wti_oil"] = pd.to_numeric(df["wti_oil"], errors="coerce").ffill(limit=5)
clean["dxy"] = pd.to_numeric(df["dxy"], errors="coerce").ffill(limit=5)

clean["vnindex"] = (
    df["vnindex"].astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
    .replace({"nan": np.nan, "None": np.nan, "": np.nan})
)
clean["vnindex"] = pd.to_numeric(clean["vnindex"], errors="coerce").ffill(limit=5)

# Drop holiday row 2026-01-01 if present
clean = clean[clean["Date"] < "2026-01-01"].copy()

print("Clean shape:", clean.shape)
print("\nMissing values:")
print(clean.isna().sum())

clean.head()


Clean shape: (4021, 7)

Missing values:
Date       0
usd_vnd    0
vix        0
us_10y     0
wti_oil    0
dxy        0
vnindex    0
dtype: int64


,Date,usd_vnd,vix,us_10y,wti_oil,dxy,vnindex
0,2010-01-04,18474.043968,20.04,3.85,81.52,77.529999,517.05
1,2010-01-05,18469.000000,19.35,3.77,81.74,77.620003,532.53
2,2010-01-06,18469.000000,19.16,3.85,83.12,77.489998,534.46
3,2010-01-07,18474.000000,19.06,3.85,82.60,77.910004,533.34
4,2010-01-08,18469.000000,18.13,3.83,82.74,77.470001,520.90


## **USD/VND Feature**

Main feature:
- `log_usd_vnd`
- `fx_return`
- `abs_return`
- `squared_return`
- `rolling_vol_22d`

`rolling_vol_22d` used as a simple volatility proxy, equivalent to about one month of trading.


In [3]:
df_fe = clean.copy()

df_fe["log_usd_vnd"] = np.log(df_fe["usd_vnd"])
df_fe["fx_return"] = 100 * df_fe["log_usd_vnd"].diff()
df_fe["abs_return"] = df_fe["fx_return"].abs()
df_fe["squared_return"] = df_fe["fx_return"] ** 2
df_fe["rolling_vol_22d"] = df_fe["fx_return"].rolling(window=22, min_periods=15).std()

df_fe[["Date", "usd_vnd", "log_usd_vnd", "fx_return", "abs_return", "squared_return", "rolling_vol_22d", "wti_oil"]].head(25)


,Date,usd_vnd,log_usd_vnd,fx_return,abs_return,squared_return,rolling_vol_22d,wti_oil
0,2010-01-04,18474.043968,9.824122,NaN,NaN,NaN,NaN,81.52
1,2010-01-05,18469.000000,9.823849,-0.027307,0.027307,0.000746,NaN,81.74
2,2010-01-06,18469.000000,9.823849,0.000000,0.000000,0.000000,NaN,83.12
3,2010-01-07,18474.000000,9.824120,0.027069,0.027069,0.000733,NaN,82.60
4,2010-01-08,18469.000000,9.823849,-0.027069,0.027069,0.000733,NaN,82.74
5,2010-01-11,18469.000000,9.823849,0.000000,0.000000,0.000000,NaN,82.54
6,2010-01-12,18469.000000,9.823849,0.000000,0.000000,0.000000,NaN,80.79
7,2010-01-13,18469.000000,9.823849,0.000000,0.000000,0.000000,NaN,79.66
8,2010-01-14,18474.043968,9.824122,0.027307,0.027307,0.000746,NaN,79.35
9,2010-01-15,18469.608260,9.823882,-0.024013,0.024013,0.000577,NaN,77.96


## **Financial Shocks and Domestic Control Feature**

- `vix_change`: VIX change.
- `us10y_change`: US 10Y change.
- `dxy_return`: log return of DXY.
- `vnindex_return`: log return of VNINDEX.


In [4]:
df_fe["vix_change"] = df_fe["vix"].diff()
df_fe["us10y_change"] = df_fe["us_10y"].diff()

for col in ["usd_vnd", "dxy", "vnindex"]:
    df_fe.loc[df_fe[col] <= 0, col] = np.nan
    
df_fe["dxy_return"] = 100 * np.log(df_fe["dxy"]).diff()
df_fe["vnindex_return"] = 100 * np.log(df_fe["vnindex"]).diff()


In [5]:
df_fe["log_oil"] = np.log(df_fe["wti_oil"])
df_fe["oil_return"] = 100 * df_fe["log_oil"].diff()
df_fe["abs_oil_return"] = df_fe["oil_return"].abs()
df_fe["squared_oil_return"] = df_fe["oil_return"] ** 2

df_fe[[
    "Date",
    "vix", "vix_change",
    "us_10y", "us10y_change",
    "dxy", "dxy_return",
    "vnindex", "vnindex_return", "wti_oil", "oil_return", "abs_oil_return", "squared_oil_return"
]].head()

/Users/klinhfhm/Documents/Seminar 6/Time Series/final/final-time-series/.venv/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,Date,vix,vix_change,us_10y,us10y_change,dxy,dxy_return,vnindex,vnindex_return,wti_oil,oil_return,abs_oil_return,squared_oil_return
0,2010-01-04,20.04,NaN,3.85,NaN,77.529999,NaN,517.05,NaN,81.52,NaN,NaN,NaN
1,2010-01-05,19.35,-0.69,3.77,-0.08,77.620003,0.116022,532.53,2.949965,81.74,0.269509,0.269509,0.072635
2,2010-01-06,19.16,-0.19,3.85,0.08,77.489998,-0.167629,534.46,0.361766,83.12,1.674187,1.674187,2.802902
3,2010-01-07,19.06,-0.10,3.85,0.00,77.910004,0.540549,533.34,-0.209777,82.60,-0.627567,0.627567,0.393840
4,2010-01-08,18.13,-0.93,3.83,-0.02,77.470001,-0.566358,520.90,-2.360103,82.74,0.169348,0.169348,0.028679


## **Crisis Dummy Feature**

Stages of biggest shocks in worldwide history.
- `gfc_2008_2009`: The 2008 Financial Crisis
- `vietnam_devaluation_2011`: Vietnam Regulation 2011
- `covid_2020`: Covid 2020
- `fed_hiking_2022_2023`: US FED Hiking
- `crisis_dummy`


In [6]:
df_fe["vietnam_fx_policy_shock_2010_2011"] = 0
df_fe["vietnam_fx_devaluation_2010"] = 0
df_fe["vietnam_fx_devaluation_2011"] = 0

# Event windows based on observed FX return jumps
event_windows = {
    "vietnam_fx_devaluation_2010": [
        ("2010-02-11", "2010-03-11"),
        ("2010-08-17", "2010-08-20"),
    ],
    "vietnam_fx_devaluation_2011": [
        ("2011-02-11", "2011-02-18"),
    ],
}

for col, windows in event_windows.items():
    for start, end in windows:
        mask = df_fe["Date"].between(pd.to_datetime(start), pd.to_datetime(end))
        df_fe.loc[mask, col] = 1
        df_fe.loc[mask, "vietnam_fx_policy_shock_2010_2011"] = 1

# Show Vietnam FX policy shock rows
display(
    df_fe[
        [
            "Date",
            "fx_return",
            "vietnam_fx_devaluation_2010",
            "vietnam_fx_devaluation_2011",
            "vietnam_fx_policy_shock_2010_2011",
        ]
    ]
    .loc[df_fe["vietnam_fx_policy_shock_2010_2011"] == 1]
    .head(80)
)
df_fe["covid_2020"] = (
    (df_fe["Date"] >= "2020-03-01") &
    (df_fe["Date"] <= "2020-12-31")
).astype(int)

df_fe["fed_hiking_2022_2023"] = (
    (df_fe["Date"] >= "2022-03-01") &
    (df_fe["Date"] <= "2023-07-31")
).astype(int)

crisis_cols = [
    "vietnam_fx_devaluation_2010",
    "vietnam_fx_devaluation_2011",
    "covid_2020",
    "fed_hiking_2022_2023",
]

df_fe["crisis_dummy"] = df_fe[crisis_cols].max(axis=1)

display(
    df_fe[["Date", "fx_return"] + crisis_cols + ["crisis_dummy"]]
    .query("crisis_dummy == 1")
    .head(80)
)

,Date,fx_return,vietnam_fx_devaluation_2010,vietnam_fx_devaluation_2011,vietnam_fx_policy_shock_2010_2011
27,2010-02-11,2.014620,1,0,1
28,2010-02-12,0.789907,1,0,1
29,2010-02-16,0.000000,1,0,1
30,2010-02-17,0.000000,1,0,1
31,2010-02-18,-2.394219,1,0,1
32,2010-02-19,0.269179,1,0,1
33,2010-02-22,0.803217,1,0,1
34,2010-02-23,-0.642057,1,0,1
35,2010-02-24,2.360806,1,0,1
36,2010-02-25,-2.253509,1,0,1


,Date,fx_return,vietnam_fx_devaluation_2010,vietnam_fx_devaluation_2011,covid_2020,fed_hiking_2022_2023,crisis_dummy
27,2010-02-11,2.014620,1,0,0,0,1
28,2010-02-12,0.789907,1,0,0,0,1
29,2010-02-16,0.000000,1,0,0,0,1
30,2010-02-17,0.000000,1,0,0,0,1
31,2010-02-18,-2.394219,1,0,0,0,1
...,...,...,...,...,...,...,...
2596,2020-05-05,-0.105467,0,0,1,0,1
2597,2020-05-06,0.011713,0,0,1,0,1
2598,2020-05-07,0.011714,0,0,1,0,1
2599,2020-05-08,-0.210650,0,0,1,0,1


## **Export Feature**


In [10]:
final_cols = [
    "Date",
    "usd_vnd", "log_usd_vnd", "fx_return", "abs_return", "squared_return", "rolling_vol_22d",
    "vix", "vix_change",
    "us_10y", "us10y_change",
    "dxy", "dxy_return",
    "vnindex", "vnindex_return",
    "wti_oil", "log_oil", "oil_return", "abs_oil_return", "squared_oil_return",
    "vietnam_fx_devaluation_2010", "vietnam_fx_devaluation_2011", "covid_2020", "fed_hiking_2022_2023", "crisis_dummy"
]

df_fe = df_fe[final_cols]

model_vars = [
    "fx_return", "abs_return", "squared_return", "rolling_vol_22d",
    "vix_change", "us10y_change", "dxy_return", "vnindex_return"
]

df_model = df_fe.dropna(subset=model_vars).reset_index(drop=True)

df_fe.to_csv("../data/processed/processed_data.csv", index=False)
df_model.to_csv("../data/processed/model_data.csv", index=False)

print("Full FE file shape:", df_fe.shape)
print("Model-ready FE file shape:", df_model.shape)
print("Model date range:", df_model["Date"].min(), "→", df_model["Date"].max())

print("\nFinal columns:")
print(df_model.columns.tolist())

print("\nMissing values in model-ready file:")
print(df_model.isna().sum())


Full FE file shape: (4021, 25)
Model-ready FE file shape: (4006, 25)
Model date range: 2010-01-26 00:00:00 → 2025-12-31 00:00:00

Final columns:
['Date', 'usd_vnd', 'log_usd_vnd', 'fx_return', 'abs_return', 'squared_return', 'rolling_vol_22d', 'vix', 'vix_change', 'us_10y', 'us10y_change', 'dxy', 'dxy_return', 'vnindex', 'vnindex_return', 'wti_oil', 'log_oil', 'oil_return', 'abs_oil_return', 'squared_oil_return', 'vietnam_fx_devaluation_2010', 'vietnam_fx_devaluation_2011', 'covid_2020', 'fed_hiking_2022_2023', 'crisis_dummy']

Missing values in model-ready file:
Date                           0
usd_vnd                        0
log_usd_vnd                    0
fx_return                      0
abs_return                     0
squared_return                 0
rolling_vol_22d                0
vix                            0
vix_change                     0
us_10y                         0
us10y_change                   0
dxy                            0
dxy_return                     0
v

In [11]:
df_fe.head()

,Date,usd_vnd,log_usd_vnd,fx_return,abs_return,squared_return,rolling_vol_22d,vix,vix_change,us_10y,...,wti_oil,log_oil,oil_return,abs_oil_return,squared_oil_return,vietnam_fx_devaluation_2010,vietnam_fx_devaluation_2011,covid_2020,fed_hiking_2022_2023,crisis_dummy
0,2010-01-04,18474.043968,9.824122,NaN,NaN,NaN,NaN,20.04,NaN,3.85,...,81.52,4.400848,NaN,NaN,NaN,0,0,0,0,0
1,2010-01-05,18469.000000,9.823849,-0.027307,0.027307,0.000746,NaN,19.35,-0.69,3.77,...,81.74,4.403543,0.269509,0.269509,0.072635,0,0,0,0,0
2,2010-01-06,18469.000000,9.823849,0.000000,0.000000,0.000000,NaN,19.16,-0.19,3.85,...,83.12,4.420285,1.674187,1.674187,2.802902,0,0,0,0,0
3,2010-01-07,18474.000000,9.824120,0.027069,0.027069,0.000733,NaN,19.06,-0.10,3.85,...,82.60,4.414010,-0.627567,0.627567,0.393840,0,0,0,0,0
4,2010-01-08,18469.000000,9.823849,-0.027069,0.027069,0.000733,NaN,18.13,-0.93,3.83,...,82.74,4.415703,0.169348,0.169348,0.028679,0,0,0,0,0


In [12]:
df_model.head()

,Date,usd_vnd,log_usd_vnd,fx_return,abs_return,squared_return,rolling_vol_22d,vix,vix_change,us_10y,...,wti_oil,log_oil,oil_return,abs_oil_return,squared_oil_return,vietnam_fx_devaluation_2010,vietnam_fx_devaluation_2011,covid_2020,fed_hiking_2022_2023,crisis_dummy
0,2010-01-26,18469.000000,9.823849,0.000000,0.000000,0.000000,0.020919,24.55,-0.86,3.65,...,74.67,4.313078,-0.307549,0.307549,0.094586,0,0,0,0,0
1,2010-01-27,18474.043968,9.824122,0.027307,0.027307,0.000746,0.021482,23.14,-1.41,3.66,...,73.64,4.299188,-1.389005,1.389005,1.929335,0,0,0,0,0
2,2010-01-28,18469.608260,9.823882,-0.024013,0.024013,0.000577,0.021600,23.73,0.59,3.68,...,73.62,4.298917,-0.027163,0.027163,0.000738,0,0,0,0,0
3,2010-01-29,18474.043968,9.824122,0.024013,0.024013,0.000577,0.021795,24.62,0.89,3.63,...,72.85,4.288403,-1.051420,1.051420,1.105483,0,0,0,0,0
4,2010-02-01,18469.000000,9.823849,-0.027307,0.027307,0.000746,0.022088,22.59,-2.03,3.68,...,74.41,4.309590,2.118781,2.118781,4.489232,0,0,0,0,0
